# Introduction
- Date: 20241218
- Late Update: 20250504
- Author: BL
- This is a script to make polished panels for Figure 4 of the spatial lung manuscript

Update Log
- Swapped to using linear regression for gene selection over pseudotime
- Swapped this figure to be figure 4
- Took out pseudotime and replaced with neighborhood enrichment and DE

In [1]:
import pickle
import os
import itertools
import scipy
import csv
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import scvelo as scv
import gseapy as gp
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.pyplot import rc_context
from matplotlib.colors import LogNorm, Normalize
from scipy import sparse, stats
from copy import copy
from scipy.stats import norm
from sklearn.neighbors import KernelDensity
from typing import Tuple
import warnings
from matplotlib.colors import LinearSegmentedColormap, Normalize

/usr/local/python/3.12.3-mmchan/lib/python3.12/site-packages/scvi/__init__.py:31: DeprecationWarning: scvi is deprecated, please uninstall scvi via `pip uninstall scvi` and install the new scvi-tools package at github.com/YosefLab/scvi-tools
  warnings.warn(deprecation_msg, DeprecationWarning)


In [2]:
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.settings.verbosity = 0

output_loc = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/panels/Fig4/'
sup_loc = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/panels/FigS4/'

# Load helpful functions and objects

In [4]:
all_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science', 'E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
dbit_datasets = ['E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
ref_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science']
timepoints = ['E11.5', 'E12.5', 'E13.5']
cellstates = ['MesenchymeProgenitor', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'Precartilage', 'Pericyte',
              'Mesothelium', 'Epithelial(Distal)', 'Epithelial(Proximal)', 'VascularEndothelial',
              'Immune', 'Blood', 'Neuron', 'Heart']
epi_subcluster_cellstates = ['Distal', 'DistalIntermediate', 'ProximalIntermediate', 'Proximal']
mes_subcluster_cellstates = ['MP1', 'MP2', 'MP3', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'MatrixMesenchymeProgenitor', 'Precartilage', 'Pericyte']

# figure out color scheme
clusterColors = {'Neuron': 'DarkGreen',
                 'MatrixMesenchyme': 'DarkRed',
                 'MesenchymeProgenitor': 'RoyalBlue',
                 'Epithelial(Distal)': 'SkyBlue',
                 'Epithelial(Proximal)': 'Red',
                 'DbitEpithelialBridge': 'Cyan',
                 'SmoothMuscleProgenitor': 'Magenta',
                 'SmoothMuscle': 'Purple',
                 'VascularEndothelial': 'gold',
                 'Mesothelium': 'limegreen',
                 'Precartilage': 'darkgoldenrod',
                 'Immune': 'DarkOrange',
                 'Heart': 'Gray',
                 'Blood': 'tan',
                 'Pericyte': 'lightpink'}

epiSubclusterColors = {'Proximal': 'Red',
                    'ProximalIntermediate':'darkred',
                    'DistalIntermediate':'darkblue',
                    'Distal':'SkyBlue'}


mesSubclusterColors = {'MP1': 'RoyalBlue',
                       'MP2':'lawngreen',
                       'MP3':'Orange',
                       'MP4':'salmon',
                       'MatrixMesenchyme':'DarkRed',
                       'MatrixMesenchymeProgenitor': 'Red',
                       'SmoothMuscleProgenitor': 'Magenta',
                       'SmoothMuscle':'Purple',
                       'Pericyte': 'lightpink',
                       'Precartilage':'darkgoldenrod'}

phaseDict = {'G1': 'Blue', 'G2M': 'Orange', 'S': 'Green'}

greens = copy(mpl.cm.Greens)
greens.set_under("lightgray")
reds = copy(mpl.cm.Reds(np.linspace(0,1,40)))
reds = mpl.colors.ListedColormap(reds[5:])
reds.set_under("lightgray")

# Add silver as the lower color for the expression colormap
colors = ["silver", "red"]  # Colors for the gradient
expr_cmap = LinearSegmentedColormap.from_list("silver_to_red", colors)
expr_cmap.set_over("red") 
norm = Normalize(vmin=0, vmax=10)

In [5]:
def plot_spatial_groups(adata, 
                        meta_col,
                        background_adata = False, 
                        palette = False, 
                        show = True, 
                        output_loc = False, 
                        name = 'spatial_plot', 
                        size = 20):
    '''
    input
        - adata: An anndata object. Assumes 14 dbit datasets (split by 'orig.ident'), labeled as dbit in the 'tech' column. Assumes there is an x_loc and y_loc col
        - background_adata: A second anndata object that contains all the cells to plot in the background (if adata is a subset).
            if not provided then it will take adata as the background adata.
        - meta_col: A column in the .obs table or gene in the .var table
        - palette: A dictionary that contains a desired color for each value in the meta_col column or a cmap for a quantitative meta_col.
        - show: True or False on whether to show the plot. Default is False
        - output_loc: Whether or not to save the plot and what folder location. Default is False. 
        - name: The name to give the output pdf if saving the plot
        - size: The dot size for the scatter plot
    returns
        creates spatial scatter plots for the 14 lung spatial datasets. Options to save and show the plot
    '''
    # check if the output_loc folder exists, return an error if not
    if output_loc != False:
        if not os.path.exists(output_loc):
            print('Error: output_loc folder does not exist')
            return
        
    # check if meta_col exists, returns an error if not
    if meta_col not in adata.obs.columns and meta_col not in adata.var.index:
        print('Error: meta_col not in .obs or .var tables')
        return
    
    if type(background_adata) == type(False):
        background_adata = adata
    
    # Make a temp object to work with
    temp_adata = adata.copy()
    
    # Add the gene expr values to the obs table if plotting a gene
    if meta_col in adata.var.index:
        temp_adata.obs[meta_col] = temp_adata[:, meta_col].X.todense()
    
    timepoints = ['E11.5', 'E12.5', 'E13.5']
    fig, axs = plt.subplots(3, 5, figsize = (35, 21))
    row_count = 0
    for time in timepoints:
        datasets = temp_adata.obs[(temp_adata.obs['timepoint'] == time) & (temp_adata.obs['tech'] == 'DBiT_seq')]['orig.ident'].unique()

        col_count = 0
        for exp in datasets:
            temp_ax = axs[row_count, col_count]
            col_count += 1
            
            # Plot full structure in gray from the background adata
            all_cell_locs = background_adata.obs[(background_adata.obs['orig.ident'] == exp)].index
            x = []
            y = []

            for i in all_cell_locs:
                y.append(background_adata.obs.loc[i, 'y_loc'])
                x.append(background_adata.obs.loc[i, 'x_loc'])
            # Use silver with a default alpha to match the lower bound of the expression plots
            temp_ax.scatter(x, y, color = 'silver', s = size)
        
            # plot the quantitative variable
            temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp)].index
            
            if type(temp_adata.obs[meta_col][0]) != str:
                x = [0]
                y = [0]
                z = [temp_adata.obs[temp_adata.obs['tech'] == 'DBiT_seq'][meta_col].max()]
                for i in temp_locs:
                    z.append(temp_adata.obs.loc[i, meta_col])
                    y.append(temp_adata.obs.loc[i, 'y_loc'])
                    x.append(temp_adata.obs.loc[i, 'x_loc'])
                if palette != False:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = palette)
                else:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = 'viridis')
            else:
                labels = ['background']
                state_list = temp_adata.obs[meta_col].unique().tolist()
                for state in state_list:
                    temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp) & (temp_adata.obs[meta_col] == state)].index

                    x = [0]
                    y = [0]
                    s = [size]
                    for i in temp_locs:
                        y.append(temp_adata.obs.loc[i, 'y_loc'])
                        x.append(temp_adata.obs.loc[i, 'x_loc'])
                        s.append(size)
                    labels.append(state)

                    if palette != False:
                        temp_ax.scatter(x, y, s = size, color = palette[state])
                    else:
                        temp_ax.scatter(x, y, s = size)
                    if row_count == 0 and col_count == 5:
                        temp_ax.legend(labels = labels, bbox_to_anchor = (1.02, 1))
            temp_ax.set_title('{}_{}'.format(exp, meta_col))
            temp_ax.set_xlabel('X Pos')
            temp_ax.set_ylabel('Y Pos')
            temp_ax.set_xlim(0, 52)
            temp_ax.set_ylim(0, 52)

        row_count += 1
    if type(temp_adata.obs[meta_col][0]) != str:
        fig.colorbar(colorbar, ax = axs[2, 4])
    if output_loc != False:
        plt.savefig(output_loc + name, dpi = 300)
    if show:
        plt.show()
    plt.close()

# Load the objects

In [6]:
# Read in the full integrated adata
lung_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime.h5ad')

# Read in the mesenchyme subcluster dataset
mes_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_mesenchyme_subcluster_pseudotime.h5ad')

# Read in the scRNA only modules identified by Monocle for the smooth muscle pseudotime
sm_modules = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/smooth_muscle/Lung_smooth_muscle_scRNA_only_integrated_monocle_gene_modules_genes_above_0.1morans.txt', index_col = 0)

# Read the RNA velocity for the mesenchyme states
mes_alone = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/velocity/scvelo/objs/Mesenchyme_velocities.h5ad')

mes_states = ['MesenchymeProgenitor', 'SyntheticMesenchyme', 'SmoothMuscleProgenitor', 'SmoothMuscle',
              'Mesothelium', 'Pericyte', 'Precartilage']

In [7]:
# Convert epi_sub_modules to genes
# Build dictionary for converting GeneID to Gene names
file_path_gene = '/Genomics/chanlab/pengfei/dbit/Manuscript/lungmapping/computation manual/RNAvelocity/Ensembl_geneID_conversion_v2.txt'

# Create an empty dictionary to store the data
data_dict_gene = {}

# Open the file and read its contents
with open(file_path_gene, 'r') as file:
    for line in file:
        # Split each line into key and value using the tab separator
        key, value = line.strip().split(',')
        
        # Add key-value pairs to the dictionary
        data_dict_gene[key] = value
flipped_gene_dict = {value: key for key, value in data_dict_gene.items()}

# Build dictionary for converting GeneID to Gene names using the v3 table
file_path_gene_v3 = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/Ensembl_geneID_conversion_v3.txt'

# Create an empty dictionary to store the data
data_dict_gene_v3 = {}

# Open the file and read its contents
with open(file_path_gene_v3, 'r') as file:
    for line in file:
        # Split each line into key and value using the tab separator
        value, key = line.strip().split('\t')
        
        # Add key-value pairs to the dictionary
        data_dict_gene_v3[key] = value
flipped_gene_dict_v3 = {value: key for key, value in data_dict_gene.items()}

sm_modules['ensambl_id'] = sm_modules['id']
for i in sm_modules.index:
    if sm_modules.loc[i, 'ensambl_id'] in data_dict_gene_v3.keys():
        sm_modules.loc[i, 'id'] = data_dict_gene_v3[sm_modules.loc[i, 'ensambl_id']]

# Figure 4b
- UMAP of mesenchyme subclusters

In [9]:
with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(mes_adata, color = ['subcluster_cellstate'], wspace = 0.35, palette = mesSubclusterColors,
               show = False)
    plt.savefig(output_loc + 'Fig.4b_mesenchyme_cellstates.pdf', bbox_inches = "tight", dpi = 300)
    plt.close()

# Figure 4c
- Plot violin plots of several key genes

In [9]:
# Plot violin plots of expression for MP states
genes = ['Wnt2', 'Myocd', 'Prrx1', 'Pdgfrb', 'Sox9', 'Cenpa', 'Cenpf', 'Tcf21']
states = ['MP1', 'MP2', 'MP3', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'Precartilage', 'Pericyte']
for gene in genes:
    with rc_context({'figure.figsize': (6, 2)}):
        sc.pl.violin(mes_adata[(mes_adata.obs['tech'] == 'scRNA_seq') & (mes_adata.obs['subcluster_cellstate'].isin(states))], gene, 
                     order = states, groupby = 'subcluster_cellstate', jitter=0.4,
                     multi_panel= True, rotation = 90, show = False, palette = mesSubclusterColors)
        plt.savefig(output_loc + 'Fig.4c_scRNA_violin_{}.pdf'.format(gene), bbox_inches = "tight", dpi = 300)
        plt.close()

# Figure 4d
- Spatial map of MP1, MP2, MP3 subclusters
- Spatial map of MesenchymeProgenitor markers: ['Wnt2', 'Hoxb8', 'Hoxb5', 'Calb2']

In [35]:
plot_spatial_groups(adata = mes_adata[mes_adata.obs['subcluster_cellstate'].isin(['MP1', 'MP2', 'MP3'])],
                    meta_col = 'subcluster_cellstate',
                    background_adata = lung_adata,
                    output_loc = output_loc,
                    name = 'Fig.4d_spatial_localization_mesenchyme_progenitors.pdf',
                    palette = mesSubclusterColors,
                    show = False)

In [20]:
# Plot the spatial localization of each gene in only epithelial cells across all replicates
genes = ['Wnt2', 'Hoxb8', 'Hoxb5', 'Hoxb6', 'Calb2']

for gene in genes:
    plot_spatial_groups(adata = lung_adata,
                        meta_col = gene,
                        background_adata = lung_adata,
                        output_loc = output_loc,
                        name = 'Fig.4d_spatial_localization_{}.pdf'.format(gene),
                        palette = expr_cmap,
                        show = False)

# Figure 4e
- Marker gene volcano plot for matrix mesenchyme

In [ ]:
# Plot a volcano plot of Distal vs DI
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/marker_genes/cellstates/MatrixMesenchyme_vs_rest_scRNA_only.csv', index_col = 0)
upName = 'MatrixMesenchyme'
downName = 'not_MatrixMesenchyme'

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Ptn', 'Prrx1', 'Dcn', 'Col1a1', 'Mfap4', 'Wnt2', 'Sema5a', 'Aldh1a2', 'Nlgn1', 'Tmtc2', 'Rspo2', 'Fgf10']

fig, ax = plt.subplots(figsize = (5, 5))
#plt.style.use('default')

# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['logfoldchanges'].values
y = -np.log10(temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['pvals_adj_2']).values
plt.scatter(x = x, y = y, alpha = 0.3, s = 5, color = 'lightgray')

# highlight genes that are higher than 1 logfoldchange and higher than 1.3 -log10 adjustedpval
x_up = []
y_up = []
x_down = []
y_down = []
for count, val in enumerate(x):
    if x[count] > 1 and y[count] > 1.3:
        x_up.append(x[count])
        y_up.append(y[count])
plt.scatter(x = x_up, y = y_up, s = 10, color = clusterColors[upName])

for gene in annotate_genes:
    x = temp_df.loc[gene, 'logfoldchanges']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    plt.text(x, y + 0.5, gene) 

ax.axhline(1.3, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.axvline(1, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.axvline(-1, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('{} vs {} - DE'.format(upName, downName))
plt.savefig(output_loc + 'Fig.4e_MatrixMesenchyme_volcano_plot.pdf', dpi = 300)
#plt.show()
plt.close()

# Figure 4f
- Bar plot showing enrichr results

In [ ]:
enrichr = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/marker_genes/GO/MatrixMesenchyme_GO_Biological_Process_2025_table.txt', sep = '\t', index_col = 0)

terms = enrichr[enrichr['Adjusted P-value'] < 0.05][0:15].index.tolist()

data = [-np.log10(i) for i in enrichr.loc[terms, 'Adjusted P-value'].tolist()]
labels = terms

fig, ax = plt.subplots(figsize = (2, 3))

plt.barh(labels[::-1], data[::-1])
plt.xlabel('-log10(adj_pval)')
plt.xticks([0, 1, 2, 3])
plt.savefig(output_loc + 'Fig.4f_MatrixM_enrichr_boxplot.pdf', dpi = 300)
#plt.show()
plt.close()

# Figure 4g
- Spatial plot showing MatrixMesenchyme and Neurons

In [ ]:
# Plot the spatial localization of all cellstates across all replicates
plot_spatial_groups(adata = lung_adata[lung_adata.obs['cellstate'].isin(['MatrixMesenchyme', 'Neuron'])],
                    meta_col = 'cellstate',
                    background_adata = lung_adata,
                    output_loc = output_loc,
                    name = 'Fig.4g_spatial_localization_synthetic_mesenchyme_and_neuron.pdf',
                    palette = clusterColors,
                    show = False)

# Figure 4h
- Boxplot showing neighborhood enrichment scores

In [ ]:
# load neighborhood enrichment results
neuron_z_scores = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/nhood_enrichment/clusters/Neuron_nhood_enrichment.csv', index_col = 0)

data = []
labels = []
for i in ['Neuron', 'MatrixMesenchyme', 'Immune', 'VascularEndothelial', 'Pericyte', 'Precartilage', 'MesenchymeProgenitor']:
    if i not in ['DbitEpithelialBridge', 'Heart']:
        data.append(neuron_z_scores.loc[i].dropna())
        labels.append(i)
fig, ax = plt.subplots(figsize = (2, 3))

plt.boxplot(data, tick_labels = labels)

# Plot a horizontal line at zscore = 3
ax.axhline(3, zorder = 0, c = 'gray', lw = 1, ls = '--')

plt.title('Neuron Neighborhood Enrichment Zscores')
plt.ylabel('Zscore')
plt.xticks(rotation = 90)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.savefig(output_loc + 'Fig.4h_Neuron_nhood_enrichment.pdf', dpi = 300)
#plt.show()
plt.close()

# Extended Data
- Spatial maps of matrix mesenchyme marker genes: ['Col1a1', 'Col1a2', 'Igfbp5', 'Tmtc2', 'Mfap4', 'Col3a1', 'Dcn', 'Prrx1', 'Sox2', 'Rspo2', 'Ptn', 'Osr1']
- Violin plots of matrix mesenchyme marker genes: ['Col1a1', 'Col1a2', 'Igfbp5', 'Tmtc2', 'Mfap4', 'Col3a1', 'Dcn', 'Prrx1', 'Sox2', 'Rspo2', 'Ptn', 'Osr1']
- Volcano between Mesenchyme progenitor and Matrix mesenchyme
- Spatial plots of Pericyte and Vascular Endothelial cells
- Spatial Plots of Precartilage, Epithelial Distal and Epithelial Proximal
- Dotplot showing cell-cell interaction genes per cluster: ['Sdc3', 'Ptprz1', 'Wnt7b', 'Fzd1', 'Fzd10', 'Lrp5', 'Fzd4', 'Sox17']

In [13]:
os.chdir(sup_loc)

In [ ]:
# Plot the spatial localization of each gene's expression across all replicates
genes = ['Col1a1', 'Col1a2', 'Igfbp5', 'Tmtc2', 'Mfap4', 'Col3a1', 'Dcn', 'Prrx1', 'Sox2', 'Rspo2', 'Ptn', 'Osr1']

for gene in genes:
    plot_spatial_groups(adata = lung_adata,
                        meta_col = gene,
                        background_adata = lung_adata,
                        output_loc = sup_loc,
                        name = 'Fig.S4_spatial_localization_{}.pdf'.format(gene),
                        palette = expr_cmap,
                        show = False)

In [ ]:
# Violin plots of genes in scRNA cells
genes = ['Col1a1', 'Col1a2', 'Igfbp5', 'Tmtc2', 'Mfap4', 'Col3a1', 'Dcn', 'Prrx1', 'Sox2', 'Rspo2', 'Ptn', 'Osr1']
for gene in genes:
    with rc_context({'figure.figsize': (3, 1)}):
        sc.pl.violin(lung_adata[(lung_adata.obs['tech'] == 'scRNA_seq') & (lung_adata.obs['cellstate'].isin(mes_states))], gene, groupby = 'cellstate',
                     order = mes_states,
                     jitter=0.4, multi_panel= True, rotation = 90, palette = clusterColors, show = False)
        plt.savefig(sup_loc + 'Fig.S4_scRNA_violin_{}.pdf'.format(gene), bbox_inches = "tight", dpi = 300)
        plt.close()

In [ ]:
# Plot a volcano plot of Distal vs DI
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/clusters/gene_lists/Lung_MesenchymeProgenitor_SyntheticMesenchyme_wilcox_DE.csv', index_col = 0)
upName = 'MesenchymeProgenitor'
downName = 'SyntheticMesenchyme'

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Ptn', 'Prrx1', 'Dcn', 'Col1a1', 'Mfap4', 'Wnt2', 'Ran', 'Cenpa', 'Tmtc2']

fig, ax = plt.subplots(figsize = (5, 5))
#plt.style.use('default')

# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['manual_LogFC'].values
y = -np.log10(temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['pvals_adj_2']).values
plt.scatter(x = x, y = y, alpha = 0.3, s = 5, color = 'lightgray')

# highlight genes that are higher than 1 logfoldchange and higher than 1.3 -log10 adjustedpval
x_up = []
y_up = []
x_down = []
y_down = []
for count, val in enumerate(x):
    if x[count] > 1 and y[count] > 1.3:
        x_up.append(x[count])
        y_up.append(y[count])
    if x[count] < -1 and y[count] > 1.3:
        x_down.append(x[count])
        y_down.append(y[count])
plt.scatter(x = x_up, y = y_up, s = 10, color = clusterColors[upName])
plt.scatter(x = x_down, y = y_down, s = 10, color = clusterColors[downName])

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manual_LogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    plt.text(x, y + 0.5, gene) 

ax.axhline(1.3, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.axvline(1, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.axvline(-1, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('{} vs {} - Differential Expression'.format(upName, downName))
plt.savefig(sup_loc + 'Fig.S4_{}_vs_{}_volcano_plot.pdf'.format(upName, downName), dpi = 300)
#plt.show()
plt.close()

In [ ]:
# Plot the spatial localization of pericyte and vascular endothelial across all replicates
plot_spatial_groups(adata = lung_adata[lung_adata.obs['cellstate'].isin(['Pericyte', 'VascularEndothelial'])],
                    meta_col = 'cellstate',
                    background_adata = lung_adata,
                    output_loc = sup_loc,
                    name = 'Fig.S4_spatial_localization_pericyte_vascularendothelial.pdf',
                    palette = clusterColors,
                    show = False)

In [ ]:
# Plot the spatial localization of pericyte and vascular endothelial across all replicates
plot_spatial_groups(adata = lung_adata[lung_adata.obs['cellstate'].isin(['Precartilage', 'Epithelial(Distal)', 'Epithelial(Proximal)'])],
                    meta_col = 'cellstate',
                    background_adata = lung_adata,
                    output_loc = sup_loc,
                    name = 'Fig.S4_spatial_localization_precartilage_and_epithelial.pdf',
                    palette = clusterColors,
                    show = False)

In [ ]:
# Plot a dotplot of the primary cell-cell interaction genes
with rc_context({'figure.figsize': (3, 1)}):
    sc.pl.dotplot(lung_adata, 
              var_names =  ['Sdc3', 'Ptprz1', 'Wnt7b', 'Fzd1', 'Fzd10', 'Lrp5', 'Fzd4', 'Sox17'],
              groupby = 'cellstate', show = False)
    plt.savefig(sup_loc + 'Fig.S4_cell-cell_gene_dotplot.pdf', bbox_inches = "tight", dpi = 300)
    plt.close()

In [10]:
# Plot each subcluster individually

for cluster in ['MP1', 'MP2', 'MP3', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'SyntheticMesenchymeProgenitor', 'SyntheticMesenchyme']:
    plot_spatial_groups(adata = mes_adata[mes_adata.obs['subcluster_cellstate'] == cluster],
                        meta_col = 'subcluster_cellstate',
                        background_adata = lung_adata,
                        output_loc = sup_loc,
                        name = 'Fig.S4_spatial_localization_{}.pdf'.format(cluster),
                        palette = mesSubclusterColors,
                        show = False)

In [23]:
# Plot the spatial localization of marker genes
genes = ['Hoxa5', 'Hoxb5', 'Asz1', 'Calb2', 'Hoxb8', 'Ptch1', 'Ptch2', 'Bmp4', 'Bmp7', 'Mylk', 'Tcf21']

for gene in genes:
    plot_spatial_groups(adata = lung_adata,
                        meta_col = gene,
                        background_adata = lung_adata,
                        output_loc = sup_loc,
                        name = 'Fig.S4_spatial_localization_{}.pdf'.format(gene),
                        palette = expr_cmap,
                        show = False)

In [39]:
# Plot the spatial localization of the phase score
plot_spatial_groups(adata = lung_adata,
                        meta_col = 'phase',
                        background_adata = lung_adata,
                        output_loc = sup_loc,
                        name = 'Fig.S4_spatial_localization_phase.pdf'.format(gene),
                        palette = phaseDict,
                        show = False)

In [14]:
# Plot violin plots of expression for MP states
genes = ['Wnt2', 'Actg2', 'Acta2', 'Myocd', 'Mylk', 'Myl9', 'Hoxa5', 'Hoxb5', 'Asz1', 'Calb2', 'Hoxb8', 'Tcf21', 'Fgf10', 'Bmp7']
for gene in genes:
    with rc_context({'figure.figsize': (6, 2)}):
        sc.pl.violin(mes_adata[mes_adata.obs['tech'] == 'scRNA_seq'], gene, 
                     order = mes_subcluster_cellstates, groupby = 'subcluster_cellstate', jitter=0.4,
                     multi_panel= True, rotation = 90, show = False)
        plt.savefig(sup_loc + 'Fig.S4_scRNA_violin_{}.pdf'.format(gene), bbox_inches = "tight", dpi = 300)
        plt.close()

In [13]:
new_labels = []
for i in lung_adata.obs.index:
    if i in mes_adata.obs.index:
        new_labels.append(mes_adata.obs.loc[i, 'subcluster_cellstate'])
    else:
        new_labels.append(lung_adata.obs.loc[i, 'cellstate'])

lung_adata.obs['subcluster_cellstate'] = new_labels
mesSubclusterColors['VascularEndothelial'] = 'Blue'

# Plot the spatial localization of MP3 and vascular endothelial across all replicates
plot_spatial_groups(adata = lung_adata[lung_adata.obs['subcluster_cellstate'].isin(['MP3', 'VascularEndothelial'])],
                    meta_col = 'subcluster_cellstate',
                    background_adata = lung_adata,
                    output_loc = sup_loc,
                    name = 'Fig.S4_spatial_localization_MP3_vascularendothelial.pdf',
                    palette = mesSubclusterColors,
                    show = False)

In [41]:
# Plot the spatial localization of MP3 and MP1 across all replicates
plot_spatial_groups(adata = mes_adata[mes_adata.obs['subcluster_cellstate'].isin(['MP3', 'MP1'])],
                    meta_col = 'subcluster_cellstate',
                    background_adata = lung_adata,
                    output_loc = sup_loc,
                    name = 'Fig.S4_spatial_localization_MP1_MP3.pdf',
                    palette = mesSubclusterColors,
                    show = False)

In [66]:
# plot a volcano plot for each comparison
comparisons = [('MP1', 'MP3'), ('MP1', 'SmoothMuscleProgenitor'), ('MP1', 'SyntheticMesenchymeProgenitor'),
               ('MP2', 'MP3'), ('MP2', 'SmoothMuscleProgenitor'), ('MP2', 'SyntheticMesenchymeProgenitor'),
               ('SmoothMuscleProgenitor', 'MP3'), ('SyntheticMesenchymeProgenitor', 'SmoothMuscleProgenitor'),
               ('SyntheticMesenchymeProgenitor', 'MP3')]

for comparison in comparisons:
    upName = comparison[0]
    downName = comparison[1]
    
    temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/mesenchyme_subclusters/gene_lists/mesenchyme_subclusters_scRNA_only_{}_{}_wilcox_DE.csv'.format(upName, downName), index_col = 0)
    
    # add pseudolowest pvals for any that are too low for the computer
    temp_df['pvals_adj_2'] = temp_df['pvals_adj']
    for gene in temp_df[temp_df['pvals_adj'] == 0].index:
        temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()
    
    bot_genes = temp_df[(temp_df['pvals_adj'] < 0.05) & (temp_df['manual_LogFC'] < -1)][-10:].index.tolist()
    top_genes = temp_df[(temp_df['pvals_adj'] < 0.05) & (temp_df['manual_LogFC'] > 1)][:10].index.tolist()
    annotate_genes = top_genes + bot_genes
    
    fig, ax = plt.subplots(figsize = (5, 5))
    #plt.style.use('default')
    
    # Plot the volcano plot of the differential expression using manual LogFC
    x = temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['manual_LogFC'].values
    y = -np.log10(temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['pvals_adj_2']).values
    plt.scatter(x = x, y = y, alpha = 0.3, s = 5, color = 'lightgray')
    
    # highlight genes that are higher than 1 logfoldchange and higher than 1.3 -log10 adjustedpval
    x_up = []
    y_up = []
    x_down = []
    y_down = []
    for count, val in enumerate(x):
        if x[count] > 1 and y[count] > 1.3:
            x_up.append(x[count])
            y_up.append(y[count])
        if x[count] < -1 and y[count] > 1.3:
            x_down.append(x[count])
            y_down.append(y[count])
    plt.scatter(x = x_up, y = y_up, s = 10, color = mesSubclusterColors[upName])
    plt.scatter(x = x_down, y = y_down, s = 10, color = mesSubclusterColors[downName])
    
    for gene in annotate_genes:
        x = temp_df.loc[gene, 'manual_LogFC']
        y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
        plt.text(x, y + 0.5, gene) 
    
    ax.axhline(1.3, zorder = 0, c = 'gray', lw = 1, ls = '--')
    ax.axvline(1, zorder = 0, c = 'gray', lw = 1, ls = '--')
    ax.axvline(-1, zorder = 0, c = 'gray', lw = 1, ls = '--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.xlabel('LogFC')
    plt.ylabel('-log10(adj_pval)')
    plt.title('{} vs {} - Differential Expression'.format(upName, downName))
    plt.savefig(sup_loc + 'Fig.S4_{}_vs_{}_volcano_plot.pdf'.format(upName, downName), dpi = 300)
    #plt.show()
    plt.close()

In [19]:
# Plot the subcluster composition per timepoint in scRNA cells
time_composition = pd.DataFrame(0, columns = mes_subcluster_cellstates, index = ['E11.5', 'E12.5', 'E13.5'])
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition.loc[i, j] = len(mes_adata.obs[(mes_adata.obs['timepoint'] == i) & (mes_adata.obs['subcluster_cellstate'] == j) & (mes_adata.obs['tech'] == 'scRNA_seq')])
        
time_composition_frac = time_composition.copy()
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition_frac.loc[i, j] = time_composition.loc[i, j] / time_composition.loc[i].sum()

fig, ax = plt.subplots(figsize = (5, 5))
# Plot the composition of TLS1 in GFP+ and GFP- groups
time_composition_frac.plot(kind='bar', use_index = True, stacked=True, grid = False, color = mesSubclusterColors, ax = ax)
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.title('Composition per Timepoint scRNA')
plt.ylabel('% cells')
plt.tight_layout()
plt.savefig(sup_loc + 'Fig.S4_mes_subcluster_composition_per_timepoint_scRNA.pdf', dpi = 300)
#plt.show()
plt.close()

In [20]:
# Plot the subcluster composition per timepoint in scRNA cells
time_composition = pd.DataFrame(0, columns = mes_subcluster_cellstates, index = ['E11.5', 'E12.5', 'E13.5'])
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition.loc[i, j] = len(mes_adata.obs[(mes_adata.obs['timepoint'] == i) & (mes_adata.obs['subcluster_cellstate'] == j) & (mes_adata.obs['tech'] == 'DBiT_seq')])
        
time_composition_frac = time_composition.copy()
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition_frac.loc[i, j] = time_composition.loc[i, j] / time_composition.loc[i].sum()

fig, ax = plt.subplots(figsize = (5, 5))
# Plot the composition of TLS1 in GFP+ and GFP- groups
time_composition_frac.plot(kind='bar', use_index = True, stacked=True, grid = False, color = mesSubclusterColors, ax = ax)
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.title('Composition per Timepoint DBiT')
plt.ylabel('% cells')
plt.tight_layout()
plt.savefig(sup_loc + 'Fig.S4_mes_subcluster_composition_per_timepoint_DBiT.pdf', dpi = 300)
#plt.show()
plt.close()

In [16]:
# Plot violins with just MP1, SMP, SM
genes = ['Wnt2', 'Actg2', 'Acta2', 'Myocd', 'Mylk', 'Myl9']

for gene in genes:
    with rc_context({'figure.figsize': (4, 2)}):
        sc.pl.violin(mes_adata[(mes_adata.obs['tech'] == 'scRNA_seq') & (mes_adata.obs['subcluster_cellstate'].isin(['MP1', 'SmoothMuscleProgenitor', 'SmoothMuscle']))],
                     gene, order = ['MP1', 'SmoothMuscleProgenitor', 'SmoothMuscle'], groupby = 'subcluster_cellstate', jitter=0.4,
                     multi_panel= True, rotation = 90, show = False)
        plt.savefig(sup_loc + 'Fig.4S_scRNA_violin_{}_SM_Only.pdf'.format(gene), bbox_inches = "tight", dpi = 300)
        plt.close()